# Branch F — transition-to-work search pathway and barriers

This notebook clusters the 81 valid final-Branch-F employment records using
six SWTS indicators describing institutional search support, application and
interview intensity, search duration, and the main reported obstacle.

## Theory and equation

The ILO School-to-Work Transition Survey examines how young people search for
work, the duration and intensity of that search, use of employment services,
and barriers encountered. These are pathway indicators rather than a scale.

No search-difficulty score is invented. Equal-maximum-weight Gower distance is:

$$d(i,j)=\frac{1}{6}\sum_{h=1}^{6}\delta_{ijh}$$

Search duration retains questionnaire order. Registration, service-help type,
and main obstacle are nominal. Application and interview counts use `log1p`
before range-normalised Gower distance because they are strongly right-skewed;
raw-count sensitivity is reported separately.

In [1]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import AgglomerativeClustering, SpectralClustering
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

ROOT = Path.cwd()
VALUES_PATH = ROOT / "outputs/nysc_survey_cleaned_values.csv"
LABELS_PATH = ROOT / "outputs/nysc_survey_cleaned_labels.csv"
OUT = ROOT / "outputs/branch_f_search_pathway_clusters"
OUT.mkdir(parents=True, exist_ok=True)

CLUSTER_FEATURES = [
    "registered_as_jobseeker", "employment_service_help_f",
    "jobs_applied_before_hire", "interviews_before_hire",
    "search_duration_before_hire", "obstacle_finding_this_job",
]
FEATURE_TYPES = {
    "registered_as_jobseeker": "nominal",
    "employment_service_help_f": "nominal",
    "jobs_applied_before_hire": "continuous",
    "interviews_before_hire": "continuous",
    "search_duration_before_hire": "ordinal",
    "obstacle_finding_this_job": "nominal",
}
ORIGINAL_CLUSTER_VARIABLES = [
    "registered_as_jobseeker", "employment_service_help_f",
    "jobs_applied_before_hire", "interviews_before_hire",
    "search_duration_before_hire", "obstacle_finding_this_job",
]
DESCRIPTOR_VARIABLES = [
    "reason_refused_job_f", "acceptance_conditions_search",
    "had_min_wage_during_search", "willing_to_relocate_f",
]
ALL_SOURCE_VARIABLES = ORIGINAL_CLUSTER_VARIABLES + DESCRIPTOR_VARIABLES

RANDOM_STATE = 20260824
MIN_PROFILE_SIZE = 15
CANDIDATE_SUBSAMPLES = 50
CONSENSUS_SUBSAMPLES = 100
SUBSAMPLE_FRACTION = 0.80
ENSEMBLE_BOOTSTRAPS = 100

## 1. Rebuild and audit the strict valid employed cohort

In [2]:
values_all = pd.read_csv(VALUES_PATH)
labels_all = pd.read_csv(LABELS_PATH)
strict_mask = (
    values_all["branch"].eq("F")
    & values_all["branch_routing_source"].eq("translated_final_active_branch")
    & ~values_all["branch_integrity_issue"].astype(bool)
    & values_all["employment_status"].notna()
)
values = values_all.loc[strict_mask, [
    "respondent_id", "entry_branch", "branch", "branch_routing_source",
    "branch_path_disagreement", "branch_integrity_issue",
] + ALL_SOURCE_VARIABLES].copy()
labels = labels_all.loc[strict_mask, ["respondent_id"] + ALL_SOURCE_VARIABLES].copy()
labels = labels.rename(columns={v: f"{v}__response" for v in ALL_SOURCE_VARIABLES})
values = values.rename(columns={v: f"{v}__value" for v in ALL_SOURCE_VARIABLES})
cohort = values.merge(labels, on="respondent_id", validate="one_to_one")

validation = {
    "records": int(len(cohort)),
    "unique_ids": int(cohort["respondent_id"].nunique()),
    "wrong_branch": int(cohort["branch"].ne("F").sum()),
    "wrong_routing_source": int(
        cohort["branch_routing_source"].ne("translated_final_active_branch").sum()
    ),
    "path_disagreements_remaining": int(cohort["branch_path_disagreement"].sum()),
    "integrity_issues_remaining": int(cohort["branch_integrity_issue"].sum()),
}
assert validation == {
    "records": 81, "unique_ids": 81, "wrong_branch": 0,
    "wrong_routing_source": 0, "path_disagreements_remaining": 0,
    "integrity_issues_remaining": 0,
}
display(pd.Series(validation, name="value").to_frame())

,value
records,81
unique_ids,81
wrong_branch,0
wrong_routing_source,0
path_disagreements_remaining,0
integrity_issues_remaining,0


## 2. Screen variables and build original-response features

In [3]:
screening_rows = []
for variable in ALL_SOURCE_VARIABLES:
    response = cohort[f"{variable}__response"]
    counts = response.value_counts(dropna=False)
    screening_rows.append({
        "variable": variable,
        "role": "cluster_input" if variable in ORIGINAL_CLUSTER_VARIABLES else "profile_descriptor",
        "observed": int(response.notna().sum()),
        "missing": int(response.isna().sum()),
        "unique_observed_responses": int(response.nunique(dropna=True)),
        "dominant_response": str(counts.index[0]),
        "dominant_response_percent": 100 * float(counts.iloc[0] / len(response)),
    })
variable_screening = pd.DataFrame(screening_rows)

applications_raw = pd.to_numeric(
    cohort["jobs_applied_before_hire__value"], errors="coerce"
)
interviews_raw = pd.to_numeric(
    cohort["interviews_before_hire__value"], errors="coerce"
)
duration_code = pd.to_numeric(
    cohort["search_duration_before_hire__value"], errors="coerce"
)
features = pd.DataFrame(index=cohort.index)
features["registered_as_jobseeker"] = cohort[
    "registered_as_jobseeker__response"
].astype("string")
features["employment_service_help_f"] = cohort[
    "employment_service_help_f__response"
].astype("string")
features["jobs_applied_before_hire"] = np.log1p(
    applications_raw.where(applications_raw.between(0, 10000))
)
features["interviews_before_hire"] = np.log1p(
    interviews_raw.where(interviews_raw.between(0, 1000))
)
features["search_duration_before_hire"] = duration_code.where(
    duration_code.between(1, 6)
).map({1: 0.0, 2: 0.2, 3: 0.4, 4: 0.6, 5: 0.8, 6: 1.0})
features["obstacle_finding_this_job"] = cohort[
    "obstacle_finding_this_job__response"
].astype("string")

feature_audit = pd.DataFrame({
    "feature": CLUSTER_FEATURES,
    "type": [FEATURE_TYPES[v] for v in CLUSTER_FEATURES],
    "observed": [int(features[v].notna().sum()) for v in CLUSTER_FEATURES],
    "missing": [int(features[v].isna().sum()) for v in CLUSTER_FEATURES],
    "maximum_distance_weight": [1 / len(CLUSTER_FEATURES)] * len(CLUSTER_FEATURES),
})
assert features.notna().all().all()
display(variable_screening.round(2))
display(feature_audit)
count_consistency = pd.DataFrame({
    "respondent_id": cohort["respondent_id"],
    "applications": applications_raw,
    "interviews": interviews_raw,
})
count_consistency["interviews_exceed_applications"] = (
    count_consistency["interviews"] > count_consistency["applications"]
)
display(count_consistency.loc[count_consistency["interviews_exceed_applications"]])
display(pd.DataFrame({
    "applications": applications_raw.describe(),
    "interviews": interviews_raw.describe(),
}))


def mixed_gower(frame, columns=None):
    columns = list(frame.columns) if columns is None else list(columns)
    n = len(frame)
    numerator = np.zeros((n, n), dtype=float)
    denominator = np.zeros((n, n), dtype=float)
    for column in columns:
        feature_type = FEATURE_TYPES[column]
        if feature_type == "nominal":
            series = frame[column].astype("string")
            observed = series.notna().to_numpy()
            values = series.fillna("__MISSING__").to_numpy(dtype=str)
            difference = (values[:, None] != values[None, :]).astype(float)
        else:
            series = pd.to_numeric(frame[column], errors="coerce")
            observed = series.notna().to_numpy()
            values = series.fillna(0).to_numpy(dtype=float)
            difference = np.abs(values[:, None] - values[None, :])
            if feature_type == "continuous":
                valid = values[observed]
                observed_range = float(valid.max() - valid.min())
                difference = difference / observed_range if observed_range else np.zeros_like(difference)
        pair_observed = observed[:, None] & observed[None, :]
        numerator += np.where(pair_observed, difference, 0.0)
        denominator += pair_observed
    distance = np.divide(
        numerator, denominator, out=np.zeros_like(numerator), where=denominator > 0
    )
    np.fill_diagonal(distance, 0.0)
    return distance, denominator


D, pair_coverage = mixed_gower(features)
assert np.allclose(D, D.T) and D.min() >= 0 and D.max() <= 1
positive_distances = D[D > 0]
sigma = float(np.median(positive_distances))


def affinity(distance):
    result = np.exp(-(distance ** 2) / (2 * sigma ** 2))
    np.fill_diagonal(result, 1.0)
    return result


print("Gower distance median:", round(float(np.median(positive_distances)), 4))

,variable,role,observed,missing,unique_observed_responses,dominant_response,dominant_response_percent
0,registered_as_jobseeker,cluster_input,81,0,2,No,75.31
1,employment_service_help_f,cluster_input,81,0,4,No — none at all,79.01
2,jobs_applied_before_hire,cluster_input,81,0,19,10.0,16.05
3,interviews_before_hire,cluster_input,81,0,10,2.0,27.16
4,search_duration_before_hire,cluster_input,81,0,6,1 to 4 weeks,28.40
5,obstacle_finding_this_job,cluster_input,81,0,10,No major obstacles — the process was straightf...,34.57
6,reason_refused_job_f,profile_descriptor,36,45,9,nan,55.56
7,acceptance_conditions_search,profile_descriptor,81,0,5,"Only a job that was stable, well paid, AND mat...",55.56
8,had_min_wage_during_search,profile_descriptor,81,0,2,Yes,71.60
9,willing_to_relocate_f,profile_descriptor,81,0,5,No — I would not move,30.86


,feature,type,observed,missing,maximum_distance_weight
0,registered_as_jobseeker,nominal,81,0,0.166667
1,employment_service_help_f,nominal,81,0,0.166667
2,jobs_applied_before_hire,continuous,81,0,0.166667
3,interviews_before_hire,continuous,81,0,0.166667
4,search_duration_before_hire,ordinal,81,0,0.166667
5,obstacle_finding_this_job,nominal,81,0,0.166667


,respondent_id,applications,interviews,interviews_exceed_applications
20,nysc-25a4df09-782a-4258-8618-294c7d9b72ea,2.0,6.0,True
34,nysc-f0544c2d-0907-4a78-9a38-ad5a9755addd,0.0,2.0,True
60,nysc-ec5b9d5a-e70c-4040-9ff8-8fdf20593cd5,0.0,3.0,True
73,nysc-f330a5e5-dcd3-4298-8017-92d3a06c748f,0.0,2.0,True


,applications,interviews
count,81.000000,81.000000
mean,18.864198,3.246914
std,40.494059,4.032775
min,0.000000,0.000000
25%,2.000000,1.000000
50%,5.000000,2.000000
75%,20.000000,4.000000
max,300.000000,20.000000


Gower distance median: 0.3856


## 3. Compare candidate profile counts and algorithms

In [4]:
def fit_kmedoids(distance, k, n_init=200, max_iter=200, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    n = len(distance)
    best = None
    for _ in range(n_init):
        medoids = np.sort(rng.choice(n, size=k, replace=False))
        for _ in range(max_iter):
            labels_k = np.argmin(distance[:, medoids], axis=1)
            updated = medoids.copy()
            for cluster in range(k):
                members = np.flatnonzero(labels_k == cluster)
                if len(members) == 0:
                    nearest = distance[:, medoids].min(axis=1)
                    available = np.setdiff1d(np.argsort(nearest)[::-1], updated)
                    updated[cluster] = available[0]
                else:
                    within = distance[np.ix_(members, members)]
                    updated[cluster] = members[np.argmin(within.sum(axis=1))]
            if np.array_equal(updated, medoids):
                break
            medoids = updated
        labels_k = np.argmin(distance[:, medoids], axis=1)
        loss = float(distance[np.arange(n), medoids[labels_k]].sum())
        candidate = (loss, tuple(medoids.tolist()), medoids.copy(), labels_k.copy())
        if best is None or candidate[:2] < best[:2]:
            best = candidate
    return {"labels": best[3], "medoids": best[2], "loss": best[0]}


def medoids_from_labels(distance, labels_k):
    result = []
    for cluster in sorted(np.unique(labels_k)):
        members = np.flatnonzero(labels_k == cluster)
        within = distance[np.ix_(members, members)]
        result.append(members[np.argmin(within.sum(axis=1))])
    return np.asarray(result, dtype=int)


def subsample_stability(algorithm, k, reference_labels):
    offset = 0 if algorithm == "kmedoids" else 10000
    rng = np.random.default_rng(RANDOM_STATE + 100 * k + offset)
    sample_size = int(np.floor(SUBSAMPLE_FRACTION * len(D)))
    scores = []
    for repetition in range(CANDIDATE_SUBSAMPLES):
        idx = np.sort(rng.choice(len(D), size=sample_size, replace=False))
        D_sub = D[np.ix_(idx, idx)]
        if algorithm == "kmedoids":
            result = fit_kmedoids(D_sub, k, n_init=50, random_state=RANDOM_STATE + 1000 * k + repetition)
            medoids = idx[result["medoids"]]
        else:
            labels_sub = SpectralClustering(
                n_clusters=k, affinity="precomputed", assign_labels="kmeans",
                n_init=30, random_state=RANDOM_STATE + 2000 * k + repetition,
            ).fit_predict(affinity(D_sub))
            medoids = idx[medoids_from_labels(D_sub, labels_sub)]
        scores.append(adjusted_rand_score(reference_labels, np.argmin(D[:, medoids], axis=1)))
    return np.asarray(scores)


candidate_rows = []
candidate_stability_rows = []
full_source_solutions = {}
for k in range(2, 5):
    km = fit_kmedoids(D, k, n_init=400, random_state=RANDOM_STATE + k)
    sp = SpectralClustering(
        n_clusters=k, affinity="precomputed", assign_labels="kmeans",
        n_init=100, random_state=RANDOM_STATE + 100 + k,
    ).fit_predict(affinity(D))
    full_source_solutions[("kmedoids", k)] = km["labels"]
    full_source_solutions[("spectral", k)] = sp
    results = {}
    for algorithm, labels_k in [("kmedoids", km["labels"]), ("spectral", sp)]:
        stability = subsample_stability(algorithm, k, labels_k)
        results[algorithm] = {
            "silhouette": silhouette_score(D, labels_k, metric="precomputed"),
            "stability": float(stability.mean()),
            "p10": float(np.quantile(stability, 0.10)),
            "minimum_size": int(pd.Series(labels_k).value_counts().min()),
        }
        candidate_stability_rows.extend({
            "algorithm": algorithm, "k": k, "repetition": i + 1, "ari": score,
        } for i, score in enumerate(stability))
    candidate_rows.append({
        "k": k,
        "kmedoids_silhouette": results["kmedoids"]["silhouette"],
        "spectral_silhouette": results["spectral"]["silhouette"],
        "mean_source_silhouette": np.mean([results["kmedoids"]["silhouette"], results["spectral"]["silhouette"]]),
        "kmedoids_stability_mean": results["kmedoids"]["stability"],
        "spectral_stability_mean": results["spectral"]["stability"],
        "mean_source_stability": np.mean([results["kmedoids"]["stability"], results["spectral"]["stability"]]),
        "minimum_source_p10_stability": min(results["kmedoids"]["p10"], results["spectral"]["p10"]),
        "source_algorithm_ari": adjusted_rand_score(km["labels"], sp),
        "minimum_cluster_size_across_sources": min(results["kmedoids"]["minimum_size"], results["spectral"]["minimum_size"]),
    })

candidate_metrics = pd.DataFrame(candidate_rows)
candidate_stability = pd.DataFrame(candidate_stability_rows)

# PAC is used comparatively across candidate k values, not against an arbitrary
# universal cutoff. Each candidate uses the same number of 80% subsamples and
# the same two algorithms.
def candidate_consensus_pac(k):
    rng = np.random.default_rng(RANDOM_STATE + 25000 + k)
    n_local = len(D)
    sample_size_local = int(np.floor(SUBSAMPLE_FRACTION * n_local))
    cover_total = np.zeros((n_local, n_local), dtype=np.uint16)
    same_total = np.zeros((n_local, n_local), dtype=np.uint16)
    for repetition in range(CANDIDATE_SUBSAMPLES):
        idx = np.sort(rng.choice(n_local, size=sample_size_local, replace=False))
        D_sub = D[np.ix_(idx, idx)]
        cover = np.zeros((n_local, n_local), dtype=np.uint8)
        cover[np.ix_(idx, idx)] = 1
        km_labels = fit_kmedoids(
            D_sub, k, n_init=80,
            random_state=RANDOM_STATE + 26000 + 1000 * k + repetition,
        )["labels"]
        sp_labels = SpectralClustering(
            n_clusters=k, affinity="precomputed", assign_labels="kmeans",
            n_init=30,
            random_state=RANDOM_STATE + 27000 + 1000 * k + repetition,
        ).fit_predict(affinity(D_sub))
        for labels_sub in (km_labels, sp_labels):
            same = np.zeros((n_local, n_local), dtype=np.uint8)
            same[np.ix_(idx, idx)] = (
                labels_sub[:, None] == labels_sub[None, :]
            ).astype(np.uint8)
            cover_total += cover
            same_total += same
    consensus_k = np.divide(
        same_total, cover_total,
        out=np.zeros_like(same_total, dtype=float), where=cover_total > 0,
    )
    upper_k = consensus_k[np.triu_indices(n_local, 1)]
    return float(np.mean((upper_k > 0.10) & (upper_k < 0.90)))

candidate_metrics["pac_10_90"] = [
    candidate_consensus_pac(int(k)) for k in candidate_metrics["k"]
]
candidate_metrics["pac_rank_all_candidates"] = candidate_metrics[
    "pac_10_90"
].rank(ascending=True, method="min")
eligible = candidate_metrics.loc[
    candidate_metrics["minimum_cluster_size_across_sources"].ge(MIN_PROFILE_SIZE)
].copy()
candidate_size_eligible = not eligible.empty
ranking_pool = eligible.copy() if candidate_size_eligible else candidate_metrics.copy()
ranking_pool["rank_silhouette"] = ranking_pool["mean_source_silhouette"].rank(ascending=False, method="min")
ranking_pool["rank_stability"] = ranking_pool["mean_source_stability"].rank(ascending=False, method="min")
ranking_pool["rank_pac"] = ranking_pool["pac_10_90"].rank(ascending=True, method="min")
ranking_pool["primary_mean_rank"] = ranking_pool[[
    "rank_silhouette", "rank_stability", "rank_pac"
]].mean(axis=1)
best_rank = ranking_pool["primary_mean_rank"].min()
selected_k = int(ranking_pool.loc[
    ranking_pool["primary_mean_rank"].eq(best_rank)
].sort_values("k").iloc[0]["k"])
candidate_metrics = candidate_metrics.merge(
    ranking_pool[["k", "primary_mean_rank"]], on="k", how="left"
)
candidate_metrics["selected"] = candidate_metrics["k"].eq(selected_k)
display(candidate_metrics.round(4))
if candidate_size_eligible:
    print("Selected k:", selected_k)
else:
    print("No candidate met the minimum-size rule; k=", selected_k, "is fitted for rejection diagnostics only.")

,k,kmedoids_silhouette,spectral_silhouette,mean_source_silhouette,kmedoids_stability_mean,spectral_stability_mean,mean_source_stability,minimum_source_p10_stability,source_algorithm_ari,minimum_cluster_size_across_sources,pac_10_90,pac_rank_all_candidates,primary_mean_rank,selected
0,2,0.1646,0.3831,0.2739,0.6124,0.7643,0.6883,0.0168,0.0379,20,0.5577,3.0,1.3333,True
1,3,0.2029,0.2151,0.2090,0.7578,0.4143,0.5860,0.3630,0.3362,16,0.4722,2.0,1.6667,False
2,4,0.2158,0.2204,0.2181,0.8009,0.4327,0.6168,0.3584,0.3291,8,0.4491,1.0,NaN,False


Selected k: 2


## 4. Build and profile the consensus solution

In [5]:
rng = np.random.default_rng(RANDOM_STATE + 30000)
n = len(D)
sample_size = int(np.floor(SUBSAMPLE_FRACTION * n))
cover_matrices, same_matrices, base_rows = [], [], []
for repetition in range(CONSENSUS_SUBSAMPLES):
    idx = np.sort(rng.choice(n, size=sample_size, replace=False))
    D_sub = D[np.ix_(idx, idx)]
    cover = np.zeros((n, n), dtype=np.uint8)
    cover[np.ix_(idx, idx)] = 1
    km = fit_kmedoids(D_sub, selected_k, n_init=100, random_state=RANDOM_STATE + 40000 + repetition)["labels"]
    sp = SpectralClustering(
        n_clusters=selected_k, affinity="precomputed", assign_labels="kmeans",
        n_init=30, random_state=RANDOM_STATE + 50000 + repetition,
    ).fit_predict(affinity(D_sub))
    for algorithm, labels_sub in [("kmedoids", km), ("spectral", sp)]:
        same = np.zeros((n, n), dtype=np.uint8)
        same[np.ix_(idx, idx)] = (labels_sub[:, None] == labels_sub[None, :]).astype(np.uint8)
        cover_matrices.append(cover.copy())
        same_matrices.append(same)
        base_rows.append({
            "partition_id": len(base_rows) + 1, "algorithm": algorithm,
            "repetition": repetition + 1,
            "silhouette_in_subsample": silhouette_score(D_sub, labels_sub, metric="precomputed"),
            "minimum_cluster_size": int(pd.Series(labels_sub).value_counts().min()),
        })

cover_stack = np.stack(cover_matrices)
same_stack = np.stack(same_matrices)
coverage = cover_stack.sum(axis=0)
consensus = same_stack.sum(axis=0) / coverage
np.fill_diagonal(consensus, 1.0)
consensus_distance = 1 - consensus
np.fill_diagonal(consensus_distance, 0.0)
base_partition_diagnostics = pd.DataFrame(base_rows)
final_fit = fit_kmedoids(consensus_distance, selected_k, n_init=500, random_state=RANDOM_STATE + 60000)
raw_labels = final_fit["labels"]

raw_medoids = {}
for raw in sorted(np.unique(raw_labels)):
    members = np.flatnonzero(raw_labels == raw)
    within = consensus_distance[np.ix_(members, members)]
    raw_medoids[int(raw)] = members[np.argmin(within.sum(axis=1))]
# Give profile IDs a reproducible descriptive order: shorter search duration,
# then fewer applications and interviews first. This changes names only; it does
# not affect clustering.
raw_order = sorted(raw_medoids, key=lambda raw: (
    features.loc[raw_labels == raw, "search_duration_before_hire"].median(),
    features.loc[raw_labels == raw, "jobs_applied_before_hire"].median(),
    features.loc[raw_labels == raw, "interviews_before_hire"].median(),
))
raw_to_profile = {raw: i + 1 for i, raw in enumerate(raw_order)}
profile_id = np.asarray([raw_to_profile[int(raw)] for raw in raw_labels])
profile_names = {
    profile: f"Rejected diagnostic profile {profile}"
    for profile in sorted(np.unique(profile_id))
}

assignments = cohort[["respondent_id", "entry_branch", "branch", "branch_routing_source"]].copy()
for variable in ALL_SOURCE_VARIABLES:
    assignments[f"{variable}__value"] = cohort[f"{variable}__value"]
    assignments[f"{variable}__response"] = cohort[f"{variable}__response"]
assignments["search_pathway_profile_id"] = profile_id
assignments["search_pathway_profile_label"] = assignments["search_pathway_profile_id"].map(profile_names)
gower_sil = silhouette_samples(D, profile_id, metric="precomputed")
consensus_sil = silhouette_samples(consensus_distance, profile_id, metric="precomputed")
assignments["gower_individual_silhouette"] = gower_sil
assignments["consensus_individual_silhouette"] = consensus_sil

profile_sizes = assignments.groupby([
    "search_pathway_profile_id", "search_pathway_profile_label"
], as_index=False).agg(
    records=("respondent_id", "size"),
    low_gower_confidence_records=("gower_individual_silhouette", lambda x: int((x < 0.10).sum())),
    mean_gower_silhouette=("gower_individual_silhouette", "mean"),
    mean_consensus_silhouette=("consensus_individual_silhouette", "mean"),
)
profile_sizes["percent"] = 100 * profile_sizes["records"] / len(assignments)

response_rows = []
for variable in ALL_SOURCE_VARIABLES:
    col = f"{variable}__response"
    temp = assignments.groupby(["search_pathway_profile_id", col], dropna=False).size().rename("records").reset_index()
    temp["within_profile_percent"] = 100 * temp["records"] / temp.groupby("search_pathway_profile_id")["records"].transform("sum")
    temp = temp.rename(columns={col: "response"})
    temp.insert(1, "variable", variable)
    response_rows.append(temp)
profile_response_distributions = pd.concat(response_rows, ignore_index=True)
profile_modes = profile_response_distributions.sort_values(
    ["search_pathway_profile_id", "variable", "within_profile_percent"],
    ascending=[True, True, False],
).groupby(["search_pathway_profile_id", "variable"], as_index=False, dropna=False).head(1)
profile_numeric_medians = pd.DataFrame({
    "search_pathway_profile_id": profile_id,
    "applications": applications_raw,
    "interviews": interviews_raw,
    "search_duration_code": duration_code,
    "registered_jobseeker": cohort["registered_as_jobseeker__response"].eq("Yes").astype(int),
    "received_service_help": cohort["employment_service_help_f__response"].ne(
        "No — none at all"
    ).astype(int),
}).groupby("search_pathway_profile_id").agg(
    median_applications=("applications", "median"),
    median_interviews=("interviews", "median"),
    median_search_duration_code=("search_duration_code", "median"),
    percent_registered_jobseeker=("registered_jobseeker", lambda x: 100 * x.mean()),
    percent_received_service_help=("received_service_help", lambda x: 100 * x.mean()),
)
display(profile_sizes.round(3))
display(profile_modes.round(2))
display(profile_numeric_medians)

,search_pathway_profile_id,search_pathway_profile_label,records,low_gower_confidence_records,mean_gower_silhouette,mean_consensus_silhouette,percent
0,1,Rejected diagnostic profile 1,48,13,0.196,0.697,59.259
1,2,Rejected diagnostic profile 2,33,16,0.119,0.509,40.741


,search_pathway_profile_id,variable,response,records,within_profile_percent
103,1,acceptance_conditions_search,"Only a job that was stable, well paid, AND mat...",30,62.50
6,1,employment_service_help_f,No — none at all,39,81.25
109,1,had_min_wage_during_search,Yes,34,70.83
41,1,interviews_before_hire,2.0,13,27.08
11,1,jobs_applied_before_hire,0.0,10,20.83
70,1,obstacle_finding_this_job,No major obstacles — the process was straightf...,28,58.33
91,1,reason_refused_job_f,NaN,28,58.33
0,1,registered_as_jobseeker,No,38,79.17
57,1,search_duration_before_hire,1 to 4 weeks,21,43.75
113,1,willing_to_relocate_f,No — I would not move,17,35.42


,median_applications,median_interviews,median_search_duration_code,percent_registered_jobseeker,percent_received_service_help
search_pathway_profile_id,,,,,
1,3.0,2.0,2.0,20.833333,18.750000
2,10.0,3.0,4.0,30.303030,24.242424


## 5. Validation and sensitivity

In [6]:
confidence_rows = []
for i in range(n):
    own = profile_id[i]
    own_members = np.flatnonzero(profile_id == own)
    own_members = own_members[own_members != i]
    within = float(consensus[i, own_members].mean()) if len(own_members) else 1.0
    other = max(float(consensus[i, np.flatnonzero(profile_id == p)].mean()) for p in set(profile_id) - {own})
    confidence_rows.append({
        "respondent_id": cohort.iloc[i]["respondent_id"],
        "search_pathway_profile_id": int(own),
        "within_profile_consensus": within,
        "strongest_other_profile_consensus": other,
        "consensus_margin": within - other,
        "ambiguous_consensus_membership": (within < 0.70) or ((within - other) < 0.20),
    })
respondent_confidence = pd.DataFrame(confidence_rows)
assignments = assignments.merge(respondent_confidence.drop(columns="search_pathway_profile_id"), on="respondent_id", validate="one_to_one")

bootstrap_rng = np.random.default_rng(RANDOM_STATE + 70000)
bootstrap_rows = []
for repetition in range(ENSEMBLE_BOOTSTRAPS):
    selected = bootstrap_rng.choice(len(cover_stack), size=len(cover_stack), replace=True)
    boot_cover = cover_stack[selected].sum(axis=0)
    boot_same = same_stack[selected].sum(axis=0)
    boot_consensus = np.divide(boot_same, boot_cover, out=np.zeros_like(boot_same, dtype=float), where=boot_cover > 0)
    np.fill_diagonal(boot_consensus, 1.0)
    boot_distance = 1 - boot_consensus
    np.fill_diagonal(boot_distance, 0.0)
    boot_labels = fit_kmedoids(boot_distance, selected_k, n_init=80, random_state=RANDOM_STATE + 80000 + repetition)["labels"]
    bootstrap_rows.append({"repetition": repetition + 1, "ari_vs_full_consensus": adjusted_rand_score(profile_id, boot_labels)})
ensemble_stability = pd.DataFrame(bootstrap_rows)

full_km = full_source_solutions[("kmedoids", selected_k)]
full_sp = full_source_solutions[("spectral", selected_k)]
hac_average = AgglomerativeClustering(n_clusters=selected_k, metric="precomputed", linkage="average").fit_predict(consensus_distance)
hac_complete = AgglomerativeClustering(n_clusters=selected_k, metric="precomputed", linkage="complete").fit_predict(consensus_distance)

sensitivity_rows = []
for omitted in CLUSTER_FEATURES:
    retained = [v for v in CLUSTER_FEATURES if v != omitted]
    complete_for_retained = features[retained].notna().all(axis=1).to_numpy()
    reduced_frame = features.loc[complete_for_retained].reset_index(drop=True)
    D_reduced, reduced_coverage = mixed_gower(reduced_frame, retained)
    reduced = fit_kmedoids(D_reduced, selected_k, n_init=400, random_state=RANDOM_STATE + 90000 + CLUSTER_FEATURES.index(omitted))["labels"]
    sensitivity_rows.append({
        "omitted_feature": omitted, "retained_features": len(retained),
        "respondents_compared": int(complete_for_retained.sum()),
        "minimum_pair_coverage": int(reduced_coverage.min()),
        "gower_silhouette_reduced_kmedoids": silhouette_score(D_reduced, reduced, metric="precomputed"),
        "ari_reduced_kmedoids_vs_final_consensus": adjusted_rand_score(
            profile_id[complete_for_retained], reduced
        ),
    })
feature_sensitivity = pd.DataFrame(sensitivity_rows)

# Test the count preprocessing decision by replacing log1p applications and
# interviews with their original raw counts.
raw_count_features = features.copy()
raw_count_features["jobs_applied_before_hire"] = applications_raw
raw_count_features["interviews_before_hire"] = interviews_raw
D_raw_counts, _ = mixed_gower(raw_count_features)
raw_count_labels = fit_kmedoids(
    D_raw_counts, selected_k, n_init=400,
    random_state=RANDOM_STATE + 95000,
)["labels"]
encoding_sensitivity = pd.DataFrame([{
    "alternative_encoding": "application and interview counts left raw",
    "gower_silhouette": silhouette_score(
        D_raw_counts, raw_count_labels, metric="precomputed"
    ),
    "ari_vs_final_consensus": adjusted_rand_score(
        profile_id, raw_count_labels
    ),
}])

upper = consensus[np.triu_indices(n, 1)]
metrics = {
    "gower_silhouette": float(silhouette_score(D, profile_id, metric="precomputed")),
    "consensus_distance_silhouette": float(silhouette_score(consensus_distance, profile_id, metric="precomputed")),
    "ensemble_bootstrap_ari_mean": float(ensemble_stability["ari_vs_full_consensus"].mean()),
    "ensemble_bootstrap_ari_p10": float(ensemble_stability["ari_vs_full_consensus"].quantile(0.10)),
    "ari_vs_full_kmedoids": float(adjusted_rand_score(profile_id, full_km)),
    "ari_vs_full_spectral": float(adjusted_rand_score(profile_id, full_sp)),
    "ari_vs_hac_average": float(adjusted_rand_score(profile_id, hac_average)),
    "ari_vs_hac_complete": float(adjusted_rand_score(profile_id, hac_complete)),
    "minimum_profile_size": int(pd.Series(profile_id).value_counts().min()),
    "low_gower_confidence_percent": 100 * float(np.mean(gower_sil < 0.10)),
    "ambiguous_consensus_percent": 100 * float(assignments["ambiguous_consensus_membership"].mean()),
    "pac_10_90": float(np.mean((upper > 0.10) & (upper < 0.90))),
    "candidate_pac_rank_all": float(candidate_metrics.loc[
        candidate_metrics["k"].eq(selected_k), "pac_rank_all_candidates"
    ].iloc[0]),
    "ari_raw_count_encoding": float(
        adjusted_rand_score(profile_id, raw_count_labels)
    ),
}
quality_checks = pd.DataFrame([
    {"check": "original Gower silhouette >= 0.25", "value": metrics["gower_silhouette"], "passed": metrics["gower_silhouette"] >= 0.25},
    {"check": "ensemble bootstrap mean ARI >= 0.80", "value": metrics["ensemble_bootstrap_ari_mean"], "passed": metrics["ensemble_bootstrap_ari_mean"] >= 0.80},
    {"check": f"minimum profile size >= {MIN_PROFILE_SIZE}", "value": metrics["minimum_profile_size"], "passed": metrics["minimum_profile_size"] >= MIN_PROFILE_SIZE},
    {"check": f"minimum source-algorithm profile size >= {MIN_PROFILE_SIZE}", "value": float(candidate_metrics.loc[candidate_metrics["k"].eq(selected_k), "minimum_cluster_size_across_sources"].iloc[0]), "passed": candidate_size_eligible},
    {"check": "ARI with full K-medoids >= 0.60", "value": metrics["ari_vs_full_kmedoids"], "passed": metrics["ari_vs_full_kmedoids"] >= 0.60},
    {"check": "ARI with full spectral >= 0.60", "value": metrics["ari_vs_full_spectral"], "passed": metrics["ari_vs_full_spectral"] >= 0.60},
    {"check": "ambiguous respondents <= 20% (analyst caution rule)", "value": metrics["ambiguous_consensus_percent"], "passed": metrics["ambiguous_consensus_percent"] <= 20},
    {"check": "low-confidence respondents <= 20% (analyst caution rule)", "value": metrics["low_gower_confidence_percent"], "passed": metrics["low_gower_confidence_percent"] <= 20},
])
primary_pass = bool(quality_checks.iloc[:6]["passed"].all())
if primary_pass and quality_checks.iloc[6:]["passed"].all():
    status = "Stable, cross-method-supported search-pathway/barriers segmentation"
elif primary_pass:
    status = "Cross-method-supported with respondent-level ambiguity caution"
else:
    status = "No sufficiently robust search-pathway/barriers segmentation found"
display(pd.Series(metrics, name="value").to_frame().round(3))
display(quality_checks.round(3))
display(feature_sensitivity.round(3))
display(encoding_sensitivity.round(3))
display(candidate_metrics[["k", "pac_10_90", "pac_rank_all_candidates", "selected"]].round(3))
print("Statistical result:", status)

,value
gower_silhouette,0.165
consensus_distance_silhouette,0.621
ensemble_bootstrap_ari_mean,0.532
ensemble_bootstrap_ari_p10,0.050
ari_vs_full_kmedoids,1.000
ari_vs_full_spectral,0.038
ari_vs_hac_average,0.038
ari_vs_hac_complete,0.038
minimum_profile_size,33.000
low_gower_confidence_percent,35.802


,check,value,passed
0,original Gower silhouette >= 0.25,0.165,False
1,ensemble bootstrap mean ARI >= 0.80,0.532,False
2,minimum profile size >= 15,33.000,True
3,minimum source-algorithm profile size >= 15,20.000,True
4,ARI with full K-medoids >= 0.60,1.000,True
5,ARI with full spectral >= 0.60,0.038,False
6,ambiguous respondents <= 20% (analyst caution ...,27.160,False
7,low-confidence respondents <= 20% (analyst cau...,35.802,False


,omitted_feature,retained_features,respondents_compared,minimum_pair_coverage,gower_silhouette_reduced_kmedoids,ari_reduced_kmedoids_vs_final_consensus
0,registered_as_jobseeker,5,81,5,0.189,1.000
1,employment_service_help_f,5,81,5,0.191,0.951
2,jobs_applied_before_hire,5,81,5,0.175,0.902
3,interviews_before_hire,5,81,5,0.183,0.902
4,search_duration_before_hire,5,81,5,0.307,0.152
5,obstacle_finding_this_job,5,81,5,0.525,0.017


,alternative_encoding,gower_silhouette,ari_vs_final_consensus
0,application and interview counts left raw,0.177,0.902


,k,pac_10_90,pac_rank_all_candidates,selected
0,2,0.558,3.0,True
1,3,0.472,2.0,False
2,4,0.449,1.0,False


Statistical result: No sufficiently robust search-pathway/barriers segmentation found


## 6. Visualise and save outputs

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(candidate_metrics["k"], candidate_metrics["mean_source_silhouette"], marker="o")
axes[0].axvline(selected_k, color="tab:red", linestyle="--", label=f"selected k={selected_k}")
axes[0].axhline(0.25, color="grey", linestyle=":")
axes[0].set(title="Candidate separation", xlabel="Profiles", ylabel="Mean Gower silhouette")
axes[0].legend()
axes[1].plot(candidate_metrics["k"], candidate_metrics["mean_source_stability"], marker="o", label="mean stability")
axes[1].plot(candidate_metrics["k"], candidate_metrics["source_algorithm_ari"], marker="s", label="algorithm agreement")
axes[1].axvline(selected_k, color="tab:red", linestyle="--")
axes[1].set(title="Stability and agreement", xlabel="Profiles", ylabel="ARI", ylim=(-0.05, 1.05))
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT / "search_pathway_candidate_diagnostics.png", dpi=180, bbox_inches="tight")
plt.show()

J = np.eye(n) - np.ones((n, n)) / n
B = -0.5 * J @ (D ** 2) @ J
eigenvalues, eigenvectors = np.linalg.eigh(B)
order = np.argsort(eigenvalues)[::-1]
positive = order[eigenvalues[order] > 0][:2]
coordinates = eigenvectors[:, positive] * np.sqrt(eigenvalues[positive])
# Respondents with identical categorical response patterns have identical MDS
# coordinates. A small deterministic display-only jitter makes all records
# visible without changing clustering, distances, or validation.
plot_rng = np.random.default_rng(RANDOM_STATE + 99000)
plot_coordinates = coordinates + plot_rng.normal(0, 0.012, size=coordinates.shape)
fig, ax = plt.subplots(figsize=(8, 6))
for pid in sorted(np.unique(profile_id)):
    mask = profile_id == pid
    ax.scatter(plot_coordinates[mask, 0], plot_coordinates[mask, 1], s=42, alpha=0.7, edgecolor="white", linewidth=0.5, label=profile_names[int(pid)])
ax.set(title="Rejected diagnostic: Branch F search pathways and barriers", xlabel="Classical MDS axis 1 (jittered for display only)", ylabel="Classical MDS axis 2 (jittered for display only)")
ax.legend()
fig.text(0.5, 0.01, "Small jitter separates respondents with identical answer patterns; clustering uses the original Gower distances.", ha="center", fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "search_pathway_profile_scatterplot.png", dpi=180, bbox_inches="tight")
plt.show()

variable_screening.to_csv(OUT / "search_pathway_variable_screening.csv", index=False)
feature_audit.to_csv(OUT / "search_pathway_feature_audit.csv", index=False)
count_consistency.to_csv(OUT / "search_pathway_count_consistency_audit.csv", index=False)
candidate_metrics.to_csv(OUT / "search_pathway_candidate_metrics.csv", index=False)
candidate_stability.to_csv(OUT / "search_pathway_candidate_subsample_stability.csv", index=False)
base_partition_diagnostics.to_csv(OUT / "search_pathway_consensus_base_partitions.csv", index=False)
assignments.to_csv(OUT / "search_pathway_consensus_assignments.csv", index=False)
profile_sizes.to_csv(OUT / "search_pathway_profile_sizes.csv", index=False)
profile_response_distributions.to_csv(OUT / "search_pathway_profile_response_distributions.csv", index=False)
profile_modes.to_csv(OUT / "search_pathway_profile_modes.csv", index=False)
profile_numeric_medians.to_csv(OUT / "search_pathway_profile_numeric_medians.csv")
respondent_confidence.to_csv(OUT / "search_pathway_respondent_confidence.csv", index=False)
ensemble_stability.to_csv(OUT / "search_pathway_consensus_ensemble_stability.csv", index=False)
feature_sensitivity.to_csv(OUT / "search_pathway_leave_one_feature_out.csv", index=False)
encoding_sensitivity.to_csv(OUT / "search_pathway_encoding_sensitivity.csv", index=False)
quality_checks.to_csv(OUT / "search_pathway_quality_checks.csv", index=False)

summary = {
    "cohort": {"records": 81, "branch": "F", "strict_final_branch_records": 82, "excluded_routing_integrity_records": 1},
    "dimension": "transition-to-work search pathway and barriers",
    "clustering_features": CLUSTER_FEATURES,
    "descriptors": DESCRIPTOR_VARIABLES,
    "method": {"distance": "equal-weight mixed Gower on six questionnaire indicators; counts use log1p", "selected_k": selected_k, "ensemble": "K-medoids plus spectral consensus", "base_partitions": 200},
    "profiles": {"sizes": {str(int(r.search_pathway_profile_id)): int(r.records) for r in profile_sizes.itertuples()}, "names": {str(k): v for k, v in profile_names.items()}},
    "metrics": metrics,
    "decision": {"retained_for_clustering": False, "diagnostic_partition_only": True, "primary_checks_passed": primary_pass, "checks_passed": int(quality_checks["passed"].sum()), "checks_total": int(len(quality_checks)), "status": status},
    "limitations": [
        "These are exploratory search-pathway response profiles, not a validated search-difficulty scale.",
        "Applications and interviews are self-reported approximate counts; four records have interviews exceeding applications and are flagged but not automatically removed.",
        "Log1p count preprocessing limits domination by the applications maximum of 300; raw-count sensitivity is reported separately.",
        "Main obstacle is nominal, so no severity ordering is imposed across qualitatively different barriers.",
        "The selected diagnostic k=2 had Gower silhouette 0.165, ensemble bootstrap mean ARI 0.532, and spectral agreement ARI 0.038.",
        "Raw-versus-log count encoding agreement was ARI 0.902, so preprocessing choice does not explain the broader instability.",
        "PAC is reported comparatively across k=2 to k=4; no universal absolute PAC pass/fail cutoff is imposed.",
        "The one routing-corrupted final-Branch-F record is excluded from this employment analysis.",
    ],
}
with open(OUT / "search_pathway_clustering_summary.json", "w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)
print(json.dumps(summary, indent=2))

{
  "cohort": {
    "records": 81,
    "branch": "F",
    "strict_final_branch_records": 82,
    "excluded_routing_integrity_records": 1
  },
  "dimension": "transition-to-work search pathway and barriers",
  "clustering_features": [
    "registered_as_jobseeker",
    "employment_service_help_f",
    "jobs_applied_before_hire",
    "interviews_before_hire",
    "search_duration_before_hire",
    "obstacle_finding_this_job"
  ],
  "descriptors": [
    "reason_refused_job_f",
    "acceptance_conditions_search",
    "had_min_wage_during_search",
    "willing_to_relocate_f"
  ],
  "method": {
    "distance": "equal-weight mixed Gower on six questionnaire indicators; counts use log1p",
    "selected_k": 2,
    "ensemble": "K-medoids plus spectral consensus",
    "base_partitions": 200
  },
  "profiles": {
    "sizes": {
      "1": 48,
      "2": 33
    },
    "names": {
      "1": "Rejected diagnostic profile 1",
      "2": "Rejected diagnostic profile 2"
    }
  },
  "metrics": {
    "gowe

/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_4461/2586210603.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/53/hrc8p44527s_xqt62l2hfyvw0000gn/T/ipykernel_4461/2586210603.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Interpretation guardrails

- These profiles describe search pathways and reported barriers, not personal
  effort, employability, or responsibility for labour-market conditions.
- A high application count is search intensity, not automatically search failure.
- They do not measure personal worth, productivity, or programme eligibility.
- Profile names are descriptive and assigned only after examining distributions.